# Assignment 10 - Problem 4
## Routing support messages with the OpenAI Decisions API

**Name:** Hussein Jawad

In this problem I write a function `classify_support_message(message)` that uses `client.decisions.create()` to decide which department should handle a customer-support message. It asks one **choice** question with four options (Billing, Technical support, Account access, Other) and returns a Python dictionary with the chosen department and the probability of each option.

Steps:
1. Setup
2. The four departments
3. The function `classify_support_message()`
4. My expected labels (written before running the function)
5. Run the function on the five messages
6. Compare my labels with the API's decisions

At the end are my observations and an appendix with the concepts.

## Step 1 - Setup

Same setup as Problems 1 and 3: install the libraries, mount my Google Drive folder (where the `.env` file with my API key is), and create the client. The Decisions API uses the model `gpt-6-luna`, like in OpenAI Notebook 2.

In [ ]:
# Install the OpenAI Python library and python-dotenv (reads the .env file)
%pip install -q --upgrade openai python-dotenv

In [ ]:
import os   # file and folder functions

PROJECT_DIR = "/content/drive/MyDrive/CSCI-E-89-1-Deep-Learning/HW10"   # Drive folder with this notebook and .env

try:                                 # works only in Colab
    from google.colab import drive   # this import only works inside Colab
    drive.mount("/content/drive")    # connect my Google Drive to the Colab machine
    os.chdir(PROJECT_DIR)            # work inside the project folder so .env is found
except ImportError:                  # not in Colab
    pass                             # stay in the current folder

print("Working folder:", os.getcwd())               # show the folder we are working in
print(".env found    :", os.path.exists(".env"))    # check that the key file is there

In [ ]:
import openai                                   # the OpenAI library (used here to print its version)
from dotenv import load_dotenv, find_dotenv     # read variables from the .env file
from openai import OpenAI                       # the client class used for all API calls

load_dotenv(find_dotenv(usecwd=True), override=True)   # load OPENAI_API_KEY from .env

DECISION_MODEL = "gpt-6-luna"   # model used with the Decisions API in the lab
client = OpenAI()               # create the client; it reads OPENAI_API_KEY from the environment

key = os.getenv("OPENAI_API_KEY")                                 # get the key that was loaded
assert key, "OPENAI_API_KEY not found - check the .env file"      # stop here if the key is missing
print("openai version:", openai.__version__)                      # library version
print("Key           :", key[:3] + "..." + key[-4:])              # masked key, never the full key

## Step 2 - The four departments

These are the four options from the assignment. Each option has a `value` (the name the API returns) and a `description` that tells the model when to choose it. "Other" is the fallback for messages that fit none of the departments.

Message 5 in the test set involves two departments (it can't log in, and it keeps getting charged), so I also tell the model what to do in that case: choose the department that can fix the problem that is blocking the customer.

In [ ]:
DEPARTMENTS = [                                                                              # the four options
    {"value": "Billing",           "description": "Payments, invoices, refunds, and unexpected charges."},
    {"value": "Technical support", "description": "Software errors or problems using the product."},
    {"value": "Account access",    "description": "Login, password, and authentication problems."},
    {"value": "Other",             "description": "Requests outside the categories above."},
]

INSTRUCTIONS = ("Which department should handle this customer-support message? "         # the question
                "If the message involves more than one department, choose the department "  # how to handle overlap
                "that can fix the problem that is blocking the customer.")

## Step 3 - The function `classify_support_message()`

The function sends the message to `client.decisions.create()` with one choice question named `department`. The API sends back a typed answer instead of free text:
- `answer.choice` - the chosen department (always one of my four values)
- `answer.probabilities` - the probability of each option
- `answer.confidence` - how sure the API is about the choice

As in the lab, I find the answer by the question's name and handle the case where the API refuses to answer. The function returns a Python dictionary.

In [ ]:
def classify_support_message(message):
    # Ask the Decisions API which department should handle the message
    result = client.decisions.create(
        model=DECISION_MODEL,                       # gpt-6-luna
        input=message,                              # the customer's message is the evidence
        questions=[{                                # one question ...
            "type": "choice",                       # ... a single choice question
            "name": "department",                   # stable name, echoed back in the answer
            "instructions": INSTRUCTIONS,           # what to decide
            "choices": DEPARTMENTS,                 # the four options with descriptions
        }],
    )
    answer = next(a for a in result.answers if a.name == "department")   # find my question's answer by its name

    if answer.type == "refusal":                    # the API refused to judge this message
        return {"message": message, "department": None, "confidence": None,     # no decision
                "probabilities": {}, "refused": True}

    return {"message": message,                                                  # the message
            "department": answer.choice,                                         # the chosen department
            "confidence": answer.confidence,                                     # how sure the API is
            "probabilities": {p.value: p.probability for p in answer.probabilities}}   # probability of every option

## Step 4 - My expected labels (before running the function)

I wrote down which department I expect for each message **before** calling the API, so the API's answers can't influence my labels:

| # | Message | My expected label | Why |
|---|---|---|---|
| 1 | I was charged twice for my subscription this month. | Billing | an unexpected (double) charge |
| 2 | The application crashes whenever I upload a PDF. | Technical support | a software error |
| 3 | I lost my phone and cannot receive my login verification code. | Account access | a login / authentication problem |
| 4 | Do you offer discounts for university students? | Other | a question about pricing, not a payment, invoice, refund or charge |
| 5 | I cannot log in to cancel my subscription, and I keep getting charged. | Account access | mixed: billing and login. The charges continue *because* the customer can't log in to cancel, so the login problem blocks everything else. Billing would also be reasonable. |

In [ ]:
messages = [                                                                       # the test messages from the assignment
    "I was charged twice for my subscription this month.",
    "The application crashes whenever I upload a PDF.",
    "I lost my phone and cannot receive my login verification code.",
    "Do you offer discounts for university students?",
    "I cannot log in to cancel my subscription, and I keep getting charged.",
]

expected = ["Billing", "Technical support", "Account access", "Other", "Account access"]   # my labels, fixed before running

## Step 5 - Run the function on the five messages

For each message I print the dictionary that `classify_support_message()` returns.

In [ ]:
from pprint import pprint   # print dictionaries in a readable way

results = []                                        # one dictionary per message
for i, msg in enumerate(messages, start=1):         # go through the five messages
    r = classify_support_message(msg)               # call the Decisions API
    results.append(r)                               # keep the result
    print(f"Message {i}:")                          # which message this is
    pprint(r, sort_dicts=False)                     # the returned dictionary
    print()                                         # empty line between messages

## Step 6 - Compare my labels with the API's decisions

In [ ]:
import pandas as pd   # table for the comparison

table = pd.DataFrame({                                                            # one row per message
    "Message": messages,                                                          # the message
    "Expected": expected,                                                         # my label
    "API decision": [r["department"] for r in results],                           # the API's choice
    "Match": [r["department"] == e for r, e in zip(results, expected)],           # do they agree?
    "Confidence": [r["confidence"] for r in results],                             # how sure the API was
})
for d in DEPARTMENTS:                                                             # one column per department
    table["P(" + d["value"] + ")"] = [r["probabilities"].get(d["value"]) for r in results]   # its probability

table.index = range(1, len(table) + 1)                                            # number the rows 1-5
print(f"Agreement: {table['Match'].sum()} of {len(table)} messages")              # how many agree
table.style.format({c: "{:.0%}" for c in table.columns if c.startswith("P(") or c == "Confidence"}, na_rep="-")   # show as %

## Appendix - Concepts used in this problem

### 1. Decisions API vs Responses API

| | Responses API (`client.responses.create`) | Decisions API (`client.decisions.create`) |
|---|---|---|
| What I ask | a prompt, e.g. "Which department? Answer with one word." | a typed question: `predicate`, `choice` or `score` |
| What comes back | free text that I have to read and parse | a typed answer: the choice, plus numbers |
| Can it answer something outside my list? | yes (typos, extra words, a new category) | no, `choice` is always one of my values |
| Probabilities | no | yes, one per option |
| Good for | writing, explaining, generating | classification, routing, scoring |

The assignment requires the Decisions API because classification is exactly what it is built for: the answer is always one of my four departments, and I get a probability for each one instead of only a word.

### 2. Question types

| Type | Use | Main result |
|---|---|---|
| `predicate` | is a condition true? (e.g. "Is a refund requested?") | `probability` from 0 to 1 |
| **`choice`** | **pick one option from a list** | **`choice`, `probabilities`, `confidence`** |
| `score` | rate on ordered levels (e.g. 1-5) | `score` |

### 3. Designing a choice question (from the lab)

- **Closed set of options:** the API can only answer with one of my `value`s.
- **Descriptions:** tell the model when each option applies. I used the descriptions from the assignment.
- **A fallback option ("Other"):** for messages that fit none of the departments, so the model isn't forced into a wrong one.
- **Overlapping categories:** say what to do when a message fits more than one option. My instructions say: choose the department that can fix the problem blocking the customer.
- **A stable `name`:** the API echoes it in the answer, so I can find the right answer with `a.name == "department"`.

### 4. The answer

| Field | What it means |
|---|---|
| `answer.type` | `"choice"` for a normal answer, `"refusal"` if the API refused to judge the input |
| `answer.choice` | the chosen option's `value` |
| `answer.probabilities` | a list of `(value, probability)`, one per option; they add up to about 1 |
| `answer.confidence` | how sure the API is about its choice (0 to 1) |

**Probability vs confidence:** the probabilities show how the API spreads its belief over the options (e.g. 70% Account access, 30% Billing). Confidence is one number for the decision. The lab uses it for review routing: if confidence is below a threshold (e.g. 0.8), a person should check the message instead of trusting the automatic routing.

### 5. Evaluating a classifier

- **Write the expected labels first.** If I looked at the API's answers first, I might adjust my labels to match them.
- **Agreement** = number of messages where my label equals the API's decision.
- **Look at the probabilities, not only the choice.** A message with probabilities split between two options (like a mixed billing / login problem) is genuinely ambiguous. A disagreement there says more about the categories than about the model.